In [1]:
%load_ext jupyter_black

In [2]:
import warnings

warnings.filterwarnings("ignore")

In [3]:
import os
from dotenv import load_dotenv
from IPython.display import display, Markdown

import torch
from qdrant_client import QdrantClient
from sentence_transformers import SentenceTransformer
from langchain_openrouter import ChatOpenRouter
from langchain_core.prompts import ChatPromptTemplate

In [4]:
device = "mps" if torch.backends.mps.is_available() else "cpu"
device

'mps'

In [5]:
client = QdrantClient("localhost", port=6333)

In [6]:
load_dotenv()
hf_token = os.getenv("HF_TOKEN")
embed_model_name = "intfloat/multilingual-e5-small"
embed_model = SentenceTransformer(embed_model_name, token=hf_token, device=device)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7667.97it/s]


In [7]:
openrouter_key = os.getenv("OPENROUTER_API_KEY")
model = ChatOpenRouter(model="google/gemma-4-31b-it:free", api_key=openrouter_key)

In [8]:
query = "Yaya kuralları nelerdir?"

In [9]:
# 1. Encode with E5-Small (adding the required prefix)
query_text = f"query: {query}"
query_emb = embed_model.encode([query_text])[0]

In [10]:
# 2. Use 'query_points' instead of 'search'
results = client.query_points(
    collection_name="legal_docs", query=query_emb.tolist(), limit=5
).points

In [11]:
# 3. Extract text from the payload
context = "\n\n".join(r.payload["text"] for r in results)

In [12]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Sen bir Türk hukuku asistanısın."),
        ("user", f"Bağlam:\n{context}\n\nSoru: {query}"),
    ]
)

chain = prompt | model
response = chain.invoke({"context": context, "query": query})

In [13]:
display(Markdown(response.content))

Paylaştığınız metne göre yayaların uyması gereken kurallar şu şekildedir:

**1. Yürünecek Yerlerin Seçimi:**
*   **Genel Kural:** Yayalar; taşıt yolu bitişiğinde veya yakınında yaya yolu, banket veya alan varsa buralarda yürümek zorundadır.
*   **İstisnalar (Taşıt Yolu Üzerinde Yürüyebilecekler):**
    *   Yönetmelikteki tedbirlerin alınması şartıyla, ciddi rahatsızlık verecek boyutta eşya iten/taşıyan kişiler.
    *   Bir yetkili veya görevli yönetiminde, düzenli şekilde yürüyen yaya kafileleri (Şartlar: Taşıt yolunun en sağ şeridinden fazlasını işgal etmemek; gece veya görüşün az olduğu hallerde imkan oranında tek sıra halinde yürümek).
*   **Yaya Yolu/Banket Olmaması Durumunda:**
    *   Öncelikle bisiklet trafiğine engel olmamak şartıyla **bisiklet yolunda veya şeridinde** yürüyebilirler.
    *   Eğer bisiklet yolu/şeridi de yoksa, imkan oranında taşıt yolu kenarına yakın olmak şartıyla **taşıt yolu üzerinde** yürüyebilirler.
*   **İki Yönlü Yollar:** Her iki tarafında yaya yolu ve banket bulunmayan (veya kullanılamaz durumda olan) iki yönlü karayollarında, yaya kafileleri dışındaki yayalar **taşıt yolunun sol kenarını izlemek zorundadır.**

**2. Karşıya Geçiş Kuralları:**
*   **Yasaklar:** Taşıt yolunu; yaya geçidi, okul geçidi, kavşak giriş ve çıkışları dışında herhangi bir yerden geçmek yasaktır.
*   **Geçiş Esnasında Uyulacaklar:**
    *   Işıklı işaret varsa bu işaretlere uyulmalıdır.
    *   Işıklı işaret yoksa ve trafik yetkili kişi tarafından yönetiliyorsa, geçilecek doğrultu açıldıktan sonra yola girilmelidir.
*   **İstisna:** Eğer 100 metre kadar mesafe içerisinde yaya geçidi veya kavşak yoksa; yayalar yolu kontrol edip kendi güvenliklerini sağladıktan sonra, trafiğe engel teşkil etmeden, en kısa doğrultuda ve zamanda karşıya geçebilirler.

**3. Genel Davranış Kuralları:**
*   Yaya yollarında, geçitlerde veya zorunlu hallerde taşıt yolu üzerinde bulunan yayaların; trafiği engelleyecek, tehlikeye düşürecek davranışlarda bulunmaları veya bu alanları saygısızca kullanmaları yasaktır.

**4. Cezai Müeyyide:**
*   Bu madde hükümlerine uymayan yayalar **1.800.000 lira** para cezası ile cezalandırılırlar.